# Lesson 6: Code Search

Add `search_code` so the agent can find patterns across the codebase — the final tool in our coding assistant.

These 5 tools are the essential primitives of any coding agent:

```mermaid
graph TD
    A[User Input] --> B[LLM Agent]
    B -->|explore| C[list_files]
    B -->|read| D[read_file]
    B -->|search| E[search_code]
    B -->|execute| F[run_command]
    B -->|modify| G[edit_file]
    C & D & E & F & G -->|result| B
    B -->|text| H[Response]
    H --> A
```

> *"Less is more, folks."* — These 5 primitives are all you need. A coding agent is fundamentally an event loop executing tools based on LLM direction.

In [ ]:
%pip install openai-agents --upgrade --quiet

In [ ]:
import os
import subprocess
from getpass import getpass

from agents import Agent, Runner, function_tool

MODEL = "gpt-5.4-mini"

In [ ]:
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

In [ ]:
@function_tool
def read_file(path: str) -> str:
    """Read the contents of a file at the given path."""
    try:
        with open(path, "r") as f:
            content = f.read()
        if len(content) > 10000:
            content = content[:10000] + "\n... (truncated)"
        return content
    except FileNotFoundError:
        return f"Error: File not found: {path}"
    except Exception as e:
        return f"Error reading file: {e}"


@function_tool
def list_files(path: str = ".") -> str:
    """List all files and directories at the given path. Directories end with /."""
    try:
        entries = []
        for entry in sorted(os.listdir(path)):
            full = os.path.join(path, entry)
            if os.path.isdir(full):
                entries.append(f"{entry}/")
            else:
                entries.append(entry)
        return "\n".join(entries) if entries else "(empty directory)"
    except FileNotFoundError:
        return f"Error: Directory not found: {path}"
    except Exception as e:
        return f"Error listing files: {e}"


@function_tool
def run_command(command: str) -> str:
    """Run a shell command and return its output."""
    try:
        result = subprocess.run(command, shell=True, capture_output=True, text=True, timeout=30)
        output = result.stdout
        if result.stderr:
            output += f"\nSTDERR:\n{result.stderr}"
        if result.returncode != 0:
            output += f"\n(exit code: {result.returncode})"
        if len(output) > 10000:
            output = output[:10000] + "\n... (truncated)"
        return output if output.strip() else "(no output)"
    except subprocess.TimeoutExpired:
        return "Error: Command timed out after 30 seconds"
    except Exception as e:
        return f"Error running command: {e}"


@function_tool
def edit_file(path: str, old_str: str, new_str: str) -> str:
    """Edit a file by replacing old_str with new_str. If old_str is empty, creates the file (or appends if it exists)."""
    try:
        if old_str == "":
            os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
            mode = "a" if os.path.exists(path) else "w"
            with open(path, mode) as f:
                f.write(new_str)
            action = "Appended to" if mode == "a" else "Created"
            return f"{action} {path}"
        else:
            with open(path, "r") as f:
                content = f.read()
            count = content.count(old_str)
            if count == 0:
                return f"Error: old_str not found in {path}"
            if count > 1:
                return f"Error: old_str found {count} times in {path} (expected exactly 1)"
            content = content.replace(old_str, new_str, 1)
            with open(path, "w") as f:
                f.write(content)
            return f"Edited {path}"
    except Exception as e:
        return f"Error editing file: {e}"

## Defining the `search_code` Tool

Uses `grep` to search for patterns across files. Supports file type filtering.

In [ ]:
@function_tool
def search_code(pattern: str, path: str = ".", file_type: str = "") -> str:
    """Search for a regex pattern in code files. Returns matching lines with file paths and line numbers.
    Optionally filter by file_type (e.g., 'py', 'js', 'ts')."""
    try:
        cmd = ["grep", "-rn", "--color=never", pattern, path]
        if file_type:
            cmd.extend(["--include", f"*.{file_type}"])
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=30)
        if result.returncode == 1:
            return "No matches found."
        lines = result.stdout.strip().split("\n")[:50]
        return "\n".join(lines)
    except subprocess.TimeoutExpired:
        return "Error: Search timed out after 30 seconds"
    except Exception as e:
        return f"Error searching code: {e}"

In [ ]:
agent = Agent(
    name="Coding Assistant",
    instructions="""You are a coding assistant with full access to the codebase.

Your tools:
- list_files: Discover what files exist
- read_file: Read file contents
- search_code: Find patterns across the codebase
- run_command: Execute shell commands
- edit_file: Create or modify files

Workflow: explore first, understand the code, then make changes. Always verify your edits by running the code.""",
    model=MODEL,
    tools=[read_file, list_files, run_command, edit_file, search_code],
)

In [ ]:
result = await Runner.run(
    agent,
    "Search for all Python files that contain 'import os' in the current directory tree. How many are there?",
)
print(result.final_output)

## Summary

- Added `search_code` using grep with file type filtering and output limits.
- Our agent now has all 5 essential coding primitives: **read, list, search, run, edit**.
- These are the same primitives used by tools like Claude Code and Cursor.
- A coding agent is just ~100 lines of tool definitions running in an LLM loop.
- Next lesson: connect an MCP server (Context7) for live documentation lookup.